# LLaVA 和视觉指令微调

LLaVA是地球上被复制最多的多模态架构。它讲BLIP-2的Q-Former层替换成2层的MLP，将Flamingo的门控交叉注意力替换成了原生的词元连接，在由GPT-4生成的158k的视觉指令会话上训练。

## 问题描述

BLIP-2的Q-Former将一张图片压缩成32个Token。干净、高效、基准高，但是有两个问题。

一，Q-Former是可训练的，但是它的损失不是根据最终的任务定义的。阶段一训练ITC+ITM+ITG。阶段2训练模型损失。查询向量学到了一些用于LLM后续解码的中间表示，但是信息也在瓶颈处丢失。

二，Q-Former有188M参数，当你换LLM端时，Q-Former又得重新训。换Vision端，Q-Former也得重新训。

LLaVA的答案简单到令人尴尬：将ViT生成的576的Patch Tokens，每个过2层MLP（1024-4096-4096），然后注入到LLM的输入序列中。没有瓶颈丢失问题。没有Stage1中在奇怪的目标上训练。直接用LM损失训练MLP。

数据来源是什么呢，LLaVA的第二个洞察，使用仅文本的GPT-4来生成指令数据。将COCO（使用文本标题+物体box位置描述图片的数据集）喂给GPT-4，让它产生会话、描述、以及复杂的推理问题，没有人工标注。

## 基本概念

### 架构

LLaVA-1.5:
- 视觉编码器。 CLIP ViT-L/14@336（一阶段冻结，二阶段可选不冻结）
- 投影层。两层的MLP，激活函数为GELU。
- LLM

前向过程示例
```
image --> ViT --> 576 patches of dim 1024
patches --> MLP --> 576 tokens of dim 4096
prompt: system + "<image>" placeholder + user question

replace <image> token with 576 projected tokens

feed the full sequence to the LLM

decode response
```

### 阶段1:投影器对齐

冻结ViT，冻结LLM。只训练那个2层的MLP。数据库是558k 图片标题对。损失就是语言模型在投影的图像Token下生成的标题。

投影器学会从ViT空间映射到LLM空间，没有特定任务做监督。

### 阶段2:视觉指令为题哦啊

解冻投影器，解冻LLM（通常全量，有时LoRA）。然后在158k的视觉指令会话上训练。

视觉指令会话的来源是：
1. 取一张COCO图像
2. 抽取文本描述（5个人为写的标题+物体包围盒列表）
3. 用3个提示词模板送入GPT-4
    - 生成会话
    - 生成详细描述
    - 生成复杂的推理
4. 将GPT-4的输出作为视觉指令+回复的训练数据对。

### 为什么社区喜欢抄它

- 没有特定Stage1损失需要调整，直接用语言模型损失即可。
- 投影器只要几小时就能训好，而不是几天
- 通过重新训练投影器可以快速切换LLM
- 使用GPT-4的视觉指令数据流水线很便宜，也能应用到新的领域

### 对比Q-Former

||Q-Former|LLaVA|
|---|---|---|
|每张图片的视觉Token|32|576或2880（2x2局部图+缩放全图处理任意分辨率）|
|训练参数|188M+LLM|40M+LLM|
|阶段1损失|ITC+ITM+ITG|只有LM|
|修改LLM|需要重训|也需要重训，代价更小|
|多张图片|尴尬|自然拼接|
|视频|尴尬|自然帧拼接|
|Token预算|小|大|

MLP胜在简单和Token灵活性。Q-Former胜在Token预算。不过现在Token预算已经不构成限制，LLM的上下文已经增长到了几十万甚至百万，所以简单胜出了。

# 开始编码

教学积木：LLaVA 核心——ViT patch → 2 层 MLP 投影 → 在 `<image>` 处拼进 LLM 序列；演示阶段1（只训投影）与阶段2（解冻 LLM）的可训练参数差异。


## 1. 配置 + 冻结玩具 ViT + 2 层 MLP 投影器


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Iterator

import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class TinyLLaVAConfig:
    """小尺寸 LLaVA 示意配置（非真实 ViT-L/336）。"""

    vit_dim: int = 32
    """假 ViT 每个 patch 的通道维（真 CLIP ViT-L 为 1024）。"""

    num_patches: int = 16
    """假 patch 数（真 336/14 -> 24x24=576）。"""

    llm_dim: int = 64
    """玩具 LLM 隐层 / 词嵌入维（真 Vicuna 等常为 4096）。"""

    mlp_hidden: int = 128
    """投影器中间层维（真 LLaVA-1.5: 1024->4096->4096）。"""

    vocab_size: int = 50
    n_heads: int = 4
    n_llm_layers: int = 2
    image_token_id: int = 0
    """特殊占位 id，序列里出现它时替换为投影后的视觉 token。"""


class FrozenToyViT(nn.Module):
    """
    冻结视觉编码器示意：图像张量 -> patch 特征。

    真实 LLaVA 用 CLIP ViT-L/14@336；这里用 Conv 切 patch，权重默认冻结。
    """

    def __init__(self, cfg: TinyLLaVAConfig) -> None:
        super().__init__()
        self.cfg = cfg
        # 假定输入 (B, 3, H, H)，H*H 能整除 num_patches
        side = int(cfg.num_patches**0.5)
        assert side * side == cfg.num_patches
        self.side = side
        self.patch = nn.Conv2d(3, cfg.vit_dim, kernel_size=1)
        self.proj = nn.Linear(cfg.vit_dim, cfg.vit_dim)
        self.freeze()

    def freeze(self) -> None:
        """冻结全部参数（阶段1默认；阶段2可选解冻）。"""
        for p in self.parameters():
            p.requires_grad = False

    def unfreeze(self) -> None:
        """解冻全部参数。"""
        for p in self.parameters():
            p.requires_grad = True

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        """
        Args:
            images: 形状 ``(B, 3, H, W)`` 的图像 batch。

        Returns:
            patches: 形状 ``(B, num_patches, vit_dim)`` 的 patch 特征。
        """
        B, _, H, W = images.shape
        # 简单池化到 side x side 再 1x1 conv，保证 patch 数固定
        x = F.adaptive_avg_pool2d(images, (self.side, self.side))
        x = self.patch(x)  # (B, vit_dim, S, S)
        x = x.flatten(2).transpose(1, 2)  # (B, S*S, vit_dim)
        return self.proj(x)


class MLPProjector(nn.Module):
    """
    LLaVA-1.5 式两层 MLP 投影器：``vit_dim -> hidden -> llm_dim``，中间 GELU。

    把每个视觉 patch 映射成可与词嵌入拼接的「软 token」。
    """

    def __init__(self, vit_dim: int, hidden: int, llm_dim: int) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(vit_dim, hidden),
            nn.GELU(),
            nn.Linear(hidden, llm_dim),
        )

    def forward(self, patches: torch.Tensor) -> torch.Tensor:
        """
        Args:
            patches: ``(B, N, vit_dim)`` ViT / 冻结编码器输出。

        Returns:
            tokens: ``(B, N, llm_dim)`` 可注入 LLM 输入序列的视觉 token。
        """
        return self.net(patches)


print("FrozenToyViT / MLPProjector ready")


## 2. 在 `<image>` 占位符处插入视觉 Token + 冻结玩具 LLM


In [ ]:
def expand_image_placeholders(
    input_ids: torch.Tensor,
    image_token_id: int,
    vision_tokens: torch.Tensor,
    text_embeds: torch.Tensor,
) -> torch.Tensor:
    """
    将序列中每个 ``image_token_id`` 替换为 ``N`` 个投影视觉 token。

    假设每个样本恰好有 **1** 个 image 占位符（教学简化；多图可循环插入）。

    Args:
        input_ids: ``(B, L)`` token id，其中含一个 ``image_token_id``。
        image_token_id: 占位符 id。
        vision_tokens: ``(B, N, D)`` MLP 投影后的视觉 token。
        text_embeds: ``(B, L, D)`` 与 input_ids 对齐的词嵌入。

    Returns:
        fused: ``(B, L - 1 + N, D)`` 融合后的嵌入序列（去掉 1 个占位，换上 N 个视觉）。
    """
    B, L, D = text_embeds.shape
    N = vision_tokens.size(1)
    outs: list[torch.Tensor] = []
    for b in range(B):
        ids = input_ids[b]
        pos = (ids == image_token_id).nonzero(as_tuple=False)
        if pos.numel() != 1:
            raise ValueError(f"sample {b}: expect exactly 1 image token, got {pos.numel()}")
        i = int(pos.item())
        left = text_embeds[b, :i]           # (i, D)
        right = text_embeds[b, i + 1 :]     # (L-i-1, D)
        mid = vision_tokens[b]              # (N, D)
        outs.append(torch.cat([left, mid, right], dim=0))
    return torch.stack(outs, dim=0)


class FrozenToyLLMBlock(nn.Module):
    """单层因果 Transformer 块；默认参数冻结。"""

    def __init__(self, dim: int, n_heads: int) -> None:
        super().__init__()
        self.attn = nn.MultiheadAttention(dim, n_heads, batch_first=True)
        self.ffn = nn.Sequential(
            nn.Linear(dim, dim * 2),
            nn.GELU(),
            nn.Linear(dim * 2, dim),
        )
        self.n1 = nn.LayerNorm(dim)
        self.n2 = nn.LayerNorm(dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: ``(B, L, D)`` 隐状态。

        Returns:
            y: ``(B, L, D)`` 更新后的隐状态。
        """
        L = x.size(1)
        causal = torch.triu(
            torch.full((L, L), float("-inf"), device=x.device, dtype=x.dtype),
            diagonal=1,
        )
        h, _ = self.attn(x, x, x, attn_mask=causal, need_weights=False)
        x = self.n1(x + h)
        return self.n2(x + self.ffn(x))


class ToyCausalLM(nn.Module):
    """
    玩具因果 LLM：embedding + 若干块 + lm_head。

    阶段1冻结；阶段2可解冻（模拟 Vicuna 全量 / LoRA 前的「打开 LLM」）。
    """

    def __init__(self, cfg: TinyLLaVAConfig) -> None:
        super().__init__()
        self.cfg = cfg
        self.tok = nn.Embedding(cfg.vocab_size, cfg.llm_dim)
        self.blocks = nn.ModuleList(
            [FrozenToyLLMBlock(cfg.llm_dim, cfg.n_heads) for _ in range(cfg.n_llm_layers)]
        )
        self.head = nn.Linear(cfg.llm_dim, cfg.vocab_size, bias=False)
        self.freeze()

    def freeze(self) -> None:
        for p in self.parameters():
            p.requires_grad = False

    def unfreeze(self) -> None:
        for p in self.parameters():
            p.requires_grad = True

    def embed(self, input_ids: torch.Tensor) -> torch.Tensor:
        """
        Args:
            input_ids: ``(B, L)``。

        Returns:
            embeds: ``(B, L, llm_dim)``。
        """
        return self.tok(input_ids)

    def forward_on_embeds(self, embeds: torch.Tensor) -> torch.Tensor:
        """
        Args:
            embeds: ``(B, L', D)`` 已融合视觉后的输入嵌入。

        Returns:
            logits: ``(B, L', vocab_size)``。
        """
        x = embeds
        for block in self.blocks:
            x = block(x)
        return self.head(x)


print("expand_image_placeholders / ToyCausalLM ready")


## 3. TinyLLaVA：前向 + 阶段1/2 训练模式切换


In [ ]:
class TinyLLaVA(nn.Module):
    """
    LLaVA 最小教学版：

    ``image --ViT--> patches --MLP--> vision_tokens``
    在 ``input_ids`` 的 image 占位处拼接，再送入因果 LLM。
    """

    def __init__(self, cfg: TinyLLaVAConfig) -> None:
        super().__init__()
        self.cfg = cfg
        self.vision = FrozenToyViT(cfg)
        self.projector = MLPProjector(cfg.vit_dim, cfg.mlp_hidden, cfg.llm_dim)
        self.llm = ToyCausalLM(cfg)

    def set_stage1(self) -> None:
        """阶段1：冻 ViT + LLM，只训投影器（特征对齐）。"""
        self.vision.freeze()
        self.llm.freeze()
        for p in self.projector.parameters():
            p.requires_grad = True

    def set_stage2(self, unfreeze_vision: bool = False) -> None:
        """
        阶段2：视觉指令微调。

        Args:
            unfreeze_vision: 若为 True，同时解冻 ViT（笔记中的「可选不冻结」）。
        """
        self.llm.unfreeze()
        for p in self.projector.parameters():
            p.requires_grad = True
        if unfreeze_vision:
            self.vision.unfreeze()
        else:
            self.vision.freeze()

    def trainable_param_summary(self) -> dict[str, int]:
        """
        Returns:
            各子模块可训练参数量统计。
        """
        def count(module: nn.Module) -> int:
            return sum(p.numel() for p in module.parameters() if p.requires_grad)

        return {
            "vision": count(self.vision),
            "projector": count(self.projector),
            "llm": count(self.llm),
            "total": count(self),
        }

    def forward(
        self,
        images: torch.Tensor,
        input_ids: torch.Tensor,
    ) -> torch.Tensor:
        """
        Args:
            images: ``(B, 3, H, W)``。
            input_ids: ``(B, L)``，每行恰含 1 个 ``cfg.image_token_id``。

        Returns:
            logits: ``(B, L - 1 + N, vocab_size)``，N=``num_patches``。
        """
        patches = self.vision(images)
        vision_tokens = self.projector(patches)
        text_embeds = self.llm.embed(input_ids)
        fused = expand_image_placeholders(
            input_ids=input_ids,
            image_token_id=self.cfg.image_token_id,
            vision_tokens=vision_tokens,
            text_embeds=text_embeds,
        )
        return self.llm.forward_on_embeds(fused)

    def language_modeling_loss(
        self,
        logits: torch.Tensor,
        input_ids: torch.Tensor,
        answer_start: int,
    ) -> torch.Tensor:
        """
        在「答案段」上计算因果 LM 损失（教学简化版）。

        真实 LLaVA 会按 chat 模板精细 mask；这里假定：
        ``input_ids`` 布局为 ``[... image_placeholder ... prompt | answer ...]``，
        融合后答案从 ``answer_start + (N-1)`` 起（因 1 个占位换成 N 个视觉 token）。

        Args:
            logits: ``(B, L_fuse, V)`` 模型输出。
            input_ids: ``(B, L)`` 原始（含 1 个 image 占位）的 token。
            answer_start: 答案在 **原始** ``input_ids`` 中的起始下标。

        Returns:
            loss: 标量交叉熵。
        """
        N = self.cfg.num_patches
        # 融合后答案起点：原 answer_start 及之后整体右移 (N-1)
        ans0 = answer_start + (N - 1)
        # next-token：用 logits[:, t] 预测「融合序列」位置 t+1 的 token
        # 融合序列中答案部分对应的「文本 id」仍来自 input_ids[answer_start:]
        # 构造融合序列的 label（视觉位置 label=-100 忽略）
        B, Lf, V = logits.shape
        labels = torch.full((B, Lf), -100, device=logits.device, dtype=torch.long)
        for b in range(B):
            # 找出占位位置，重建「非视觉」token 在融合序列中的下标映射
            pos = int((input_ids[b] == self.cfg.image_token_id).nonzero(as_tuple=False).item())
            # 融合下标: [0..pos) 文本左；[pos, pos+N) 视觉；[pos+N..) 文本右
            # 原下标 j<pos -> 融合 j；j>pos -> 融合 j+(N-1)
            for j in range(input_ids.size(1)):
                if j == pos:
                    continue
                fj = j if j < pos else j + (N - 1)
                if j >= answer_start and fj < Lf:
                    labels[b, fj] = input_ids[b, j]
        # shift for causal LM
        shift_logits = logits[:, :-1, :].contiguous()
        shift_labels = labels[:, 1:].contiguous()
        return F.cross_entropy(
            shift_logits.reshape(-1, V),
            shift_labels.reshape(-1),
            ignore_index=-100,
        )


def iter_named_trainable(model: nn.Module) -> Iterator[tuple[str, nn.Parameter]]:
    """迭代可训练参数 ``(name, param)``。"""
    for n, p in model.named_parameters():
        if p.requires_grad:
            yield n, p


print("TinyLLaVA ready")


## 4. 冒烟测试：插入 shape、阶段1/2 参数、一步 LM loss


In [ ]:
def build_toy_batch(
    cfg: TinyLLaVAConfig,
    batch_size: int = 2,
    image_size: int = 32,
) -> tuple[torch.Tensor, torch.Tensor, int]:
    """
    构造玩具 batch。

    序列布局（每行）::

        [bos, image, u1, u2, a1, a2, a3]

    其中 ``image`` 为占位 id；答案从下标 4 开始。

    Args:
        cfg: 配置。
        batch_size: batch 大小。
        image_size: 方形图像边长。

    Returns:
        images: ``(B, 3, H, W)``。
        input_ids: ``(B, L)``。
        answer_start: 答案在 ``input_ids`` 中的起始下标。
    """
    B = batch_size
    images = torch.randn(B, 3, image_size, image_size)
    # ids: 0=image 占位；其它用 1..vocab-1
    rows: list[list[int]] = []
    for _ in range(B):
        rows.append(
            [
                1,
                cfg.image_token_id,
                2,
                3,
                4,
                5,
                6,
            ]
        )
    input_ids = torch.tensor(rows, dtype=torch.long)
    answer_start = 4
    return images, input_ids, answer_start


def smoke_test() -> None:
    """跑通 LLaVA 教学前向与两阶段可训练参数切换。"""
    torch.manual_seed(0)
    cfg = TinyLLaVAConfig()
    model = TinyLLaVA(cfg)
    images, input_ids, answer_start = build_toy_batch(cfg)

    print("=== shapes ===")
    patches = model.vision(images)
    vision_tokens = model.projector(patches)
    print(f"images: {tuple(images.shape)}")
    print(f"patches: {tuple(patches.shape)}  # (B, N, vit_dim)")
    print(f"vision_tokens: {tuple(vision_tokens.shape)}  # (B, N, llm_dim)")

    model.set_stage1()
    logits = model(images, input_ids)
    N = cfg.num_patches
    L = input_ids.size(1)
    print(f"logits: {tuple(logits.shape)}  # expect L-1+N = {L - 1 + N}")

    print("\n=== stage1 trainable ===")
    print(model.trainable_param_summary())
    assert model.trainable_param_summary()["vision"] == 0
    assert model.trainable_param_summary()["llm"] == 0
    assert model.trainable_param_summary()["projector"] > 0

    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-3)
    opt.zero_grad()
    loss1 = model.language_modeling_loss(logits, input_ids, answer_start)
    loss1.backward()
    opt.step()
    print(f"stage1 LM loss={loss1.item():.4f}")

    print("\n=== stage2 trainable (vision frozen) ===")
    model.set_stage2(unfreeze_vision=False)
    print(model.trainable_param_summary())
    assert model.trainable_param_summary()["llm"] > 0
    assert model.trainable_param_summary()["vision"] == 0

    opt2 = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-3)
    opt2.zero_grad()
    logits2 = model(images, input_ids)
    loss2 = model.language_modeling_loss(logits2, input_ids, answer_start)
    loss2.backward()
    opt2.step()
    print(f"stage2 LM loss={loss2.item():.4f}")

    print("\n=== stage2 trainable (vision unfrozen) ===")
    model.set_stage2(unfreeze_vision=True)
    print(model.trainable_param_summary())
    assert model.trainable_param_summary()["vision"] > 0
    print("SMOKE TEST OK")


smoke_test()
